# E1. 기준선 실험 — klue/roberta-base 로 개체명 인식(NER) 파인튜닝

**이 노트북이 하는 일**: 문장을 넣으면 개체의 글자 위치와 태그(7종)를 내는 모델을 처음으로 학습하고, 테스트셋에서 개체 단위 점수를 잰다. 이후 실험(E2 AI Hub ELECTRA, KoELECTRA 등)은 **이 노트북에서 `MODEL_NAME` 한 줄만 바꿔** 같은 조건으로 돌린다.

**전체 흐름**

```
jsonl 문장 + 정답 span
   │  ① 토크나이저: 문장 → 서브워드 토큰 + 각 토큰의 글자 위치(offset)
   ▼
토큰별 정답 꼬리표 (BIO 15개)        ← ② 정렬: 글자 위치로 span을 토큰에 입힘
   │  ③ 모델: klue/roberta-base(12층) + 분류 헤드(768→15)
   ▼
토큰별 예측 꼬리표
   │  ④ 복원: 이어지는 B/I 토큰을 합쳐 글자 위치 span으로
   ▼
예측 span  vs  정답 span   → ⑤ 개체 단위 Precision / Recall / F1
```

**실행 환경**: Colab → 런타임 → 런타임 유형 변경 → **T4 GPU**. 학습 시간은 9번 셀이 끝나면 실제로 재서 저장한다.

## 0. 패키지 설치
Colab에는 torch가 이미 있다. 토크나이저·학습 도구(transformers), 데이터 묶음(datasets), NER 채점(seqeval)만 설치한다.

In [ ]:
!pip -q install -U transformers datasets seqeval accelerate
import torch, transformers
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

## 1. 설정
실험마다 바꾸는 값은 전부 여기 모은다. **한 번에 하나만 바꾼다**는 팀 원칙을 지키려면 다른 셀은 건드리지 않는다.

- `MAX_LEN = 512`: 데이터의 "문장"은 여러 문장이 붙은 경우가 많아 길다. 소형 표본(processed_small)에서 직접 재 보니 128로 자르면 테스트 개체 604개 중 98개(16%)가 잘려 사라지고, 512로는 0개였다.
- `DATA_DIR`: Colab에서 구글 드라이브를 연결하면 이 PC 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 고친다.

In [ ]:
MODEL_NAME = "klue/roberta-base"     # E2: AI Hub ELECTRA 폴더 경로 / KoELECTRA: "monologg/koelectra-base-v3-discriminator"
EXP_NAME   = "E1_klue-roberta-base_split_v1"
MAX_LEN    = 512
LR         = 5e-5
EPOCHS     = 3
BATCH      = 16
SEED       = 42

from google.colab import drive
drive.mount("/content/drive")
PROJECT  = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
DATA_DIR = f"{PROJECT}/data/split_v1"           # 팀 공용 표본. 빠른 동작 확인만 할 땐 data/processed_small
OUT_DIR  = f"{PROJECT}/reports/results"

## 2. 데이터 불러오기
`src/prepare_data.py` 가 만든 jsonl. 한 줄이 한 문장이고 `spans` 에 정답 `[시작 글자, 끝 글자, 태그]` 가 들어 있다(끝은 포함하지 않음, 파이썬 슬라이스와 같다).

기본 데이터는 팀 공용 표본 `split_v1` 이다(train 6만·dev 1만은 학습용 TL1, test 1만은 검증용 VL1). 상용구·중복 문장을 뺀 조건이며, 만든 규칙은 `data/split_v1/stats.json` 과 Notion "데이터 전처리: 라벨 → span 변환" 페이지에 있다. 처음엔 `processed_small`(1,500문장)로 끝까지 한 번 돌려 에러가 없는지 보고, 그다음 `split_v1` 로 바꾸는 것을 권한다.

In [ ]:
import json, collections, random, numpy as np
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows, dev_rows, test_rows = (load_jsonl(f"{DATA_DIR}/{s}.jsonl") for s in ("train", "dev", "test"))
print({k: len(v) for k, v in [("train", train_rows), ("dev", dev_rows), ("test", test_rows)]})

ex = next(r for r in train_rows if r["spans"])
print(ex["text"])
for s, e, t in ex["spans"]:
    print(f"  {t}: '{ex['text'][s:e]}'  ({s},{e})")

## 3. 라벨 체계: 태그 7개 → BIO 꼬리표 15개

모델은 "개체"를 직접 내지 못하고 **토큰마다 꼬리표 하나**를 고른다. 그래서 개체를 꼬리표로 바꾼다.
- `B-태그`: 개체의 첫 토큰 (Begin)
- `I-태그`: 개체의 나머지 토큰 (Inside)
- `O`: 개체가 아닌 토큰 (Outside)

태그 7개 × (B, I) + O = **15개**. 분류 헤드의 출력 크기가 15인 이유다. B와 I를 나누는 이유는 "같은 태그 개체 두 개가 붙어 있을 때"를 구분하기 위해서다. 예: `[OG][OG]` 와 `[OG 하나]` 는 꼬리표가 `B I` 냐 `B B` 냐로 갈린다.

공식 태그 정의는 `reports/공식_태그정의_및_라벨링규칙.md` 참고(CV는 직업·직위가 아니라 **문명/문화 용어**).

In [ ]:
TAGS = ["CV", "OG", "TM", "AF", "FD", "TR", "NW"]
LABELS = ["O"] + [f"{p}-{t}" for t in TAGS for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(len(LABELS), LABELS)

## 4. 토크나이저와 라벨 정렬 (가장 중요한 전처리)

**토크나이저는 모델과 한 세트다.** `AutoTokenizer.from_pretrained(MODEL_NAME)` 로 모델을 학습할 때 쓴 어휘 사전(klue는 약 3.2만 개)을 그대로 불러온다. 토큰 번호가 모델 임베딩 행과 1:1로 연결되어 있어 다른 토크나이저로 바꾸면 안 된다.

토크나이저는 WordPiece 방식이라 의미가 아니라 사전에 있는 조각 기준으로 자른다. `##` 은 "앞 토큰에 붙은 조각"이라는 표시다.

**정렬 방법**: `return_offsets_mapping=True` 로 각 토큰이 원문 몇 번째 글자부터 몇 번째까지인지 받는다. 그 범위에
- 개체의 **첫 글자**가 들어 있으면 `B-태그`
- 개체 글자만 들어 있으면 `I-태그`
- 개체 글자가 없으면 `O`
- `[CLS]`, `[SEP]` 같은 특수 토큰은 `-100` → 손실 계산에서 빠진다(파이토치 교차 엔트로피의 ignore_index).

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def char_tags(text, spans):
    # 글자마다 (태그, 이 글자가 개체 첫 글자인가). 개체가 아니면 None
    ct = [None] * len(text)
    for s, e, tag in spans:
        for i in range(s, e):
            ct[i] = (tag, i == s)
    return ct

def encode(example):
    enc = tokenizer(example["text"], truncation=True, max_length=MAX_LEN,
                    return_offsets_mapping=True, return_token_type_ids=False)
    ct = char_tags(example["text"], example["spans"])
    labels = []
    for s, e in enc["offset_mapping"]:
        if s == e:
            labels.append(-100); continue
        inside = [ct[i] for i in range(s, e) if ct[i] is not None]
        if not inside:                              labels.append(label2id["O"])
        elif any(is_begin for _, is_begin in inside): labels.append(label2id["B-" + inside[0][0]])
        else:                                       labels.append(label2id["I-" + inside[0][0]])
    enc["labels"] = labels
    return enc

In [ ]:
# 정렬 결과 눈으로 확인: 토큰 / 글자 위치 / 꼬리표
enc = encode(ex)
print(ex["text"][:120])
for t, (s, e), l in list(zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"], enc["labels"]))[:40]:
    mark = "  ◀" if l not in (-100, 0) else ""
    print(f"{t:12s} ({s},{e})\t{id2label.get(l, '-100')}{mark}")

## 5. 복원 함수와 채점 함수

**복원(decode)**: 토큰별 꼬리표를 다시 글자 span으로 만든다. `B` 에서 새 개체를 시작하고, 같은 태그의 `I` 가 이어지면 끝 위치를 늘린다. 태그가 다른 `I` 나 앞에 개체가 없는 `I` 는 새 개체로 본다.

**채점(엄격 일치)**: 예측 span의 (시작, 끝, 태그)가 정답과 **셋 다 같아야** 맞은 것.
- Precision = 맞힌 개체 / 모델이 낸 개체 → "낸 것 중 몇 개가 맞나"
- Recall = 맞힌 개체 / 정답 개체 → "정답 중 몇 개를 찾았나"
- F1 = 두 값의 조화평균 = 2PR/(P+R). 한쪽만 높으면 점수가 안 오른다.

In [ ]:
def decode_spans(label_ids, offsets):
    spans, cur = [], None
    for lid, (s, e) in zip(label_ids, offsets):
        if s == e or lid == -100: continue
        lab = id2label[int(lid)]
        if lab == "O":
            if cur: spans.append(tuple(cur)); cur = None
            continue
        p, tag = lab.split("-", 1)
        if p == "B" or cur is None or cur[2] != tag:
            if cur: spans.append(tuple(cur))
            cur = [s, e, tag]
        else:
            cur[1] = e
    if cur: spans.append(tuple(cur))
    return spans

def span_prf(gold_list, pred_list):
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    for gold, pred in zip(gold_list, pred_list):
        g, p = set(map(tuple, gold)), set(map(tuple, pred))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    def prf(t, a, b):
        P = t / (t + a) if t + a else 0.0; R = t / (t + b) if t + b else 0.0
        return {"precision": round(P, 4), "recall": round(R, 4), "f1": round(2*P*R/(P+R), 4) if P + R else 0.0, "support": t + b}
    out = {"ALL": prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))}
    for t in TAGS: out[t] = prf(tp[t], fp[t], fn[t])
    return out

### 5-1. 점수 상한 확인: 정답을 그대로 넣으면 몇 점인가

정답 span → BIO → 다시 span 으로 되돌려 채점한다. 모델이 **완벽해도** 이 점수를 넘을 수 없다.

100%가 안 나오는 이유는 **토큰 경계와 개체 경계가 안 맞는 경우** 때문이다. 예를 들어 정답 개체가 `(73,75)` 인데 토큰이 `(73,76)` 이면, 토큰 단위로는 75에서 끊을 방법이 없다. 소형 표본에서 직접 재 보니 이런 개체가 학습 1.5%, 테스트 2.8% 정도였다. 발표에서 "이 데이터·토크나이저 조합의 이론적 상한"으로 쓸 수 있다.

In [ ]:
for name, rows in [("train", train_rows), ("dev", dev_rows), ("test", test_rows)]:
    golds, preds = [], []
    for r in rows:
        enc = encode(r)
        golds.append(r["spans"]); preds.append(decode_spans(enc["labels"], enc["offset_mapping"]))
    print(name, "상한 F1 =", span_prf(golds, preds)["ALL"])

## 6. 학습용 데이터셋 만들기
`offset_mapping` 은 학습에 필요 없어서 빼고, 테스트 복원용으로만 따로 보관한다. 길이를 맞추는 패딩은 배치마다 그 배치에서 가장 긴 문장 기준으로 한다(`DataCollatorForTokenClassification`). 짧은 문장이 대부분이라 512로 다 채우는 것보다 훨씬 빠르다.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForTokenClassification

def to_dataset(rows):
    encs = [encode(r) for r in rows]
    ds = Dataset.from_list([{"input_ids": e["input_ids"], "attention_mask": e["attention_mask"], "labels": e["labels"]} for e in encs])
    return ds, [e["offset_mapping"] for e in encs]

train_ds, _ = to_dataset(train_rows)
dev_ds, _ = to_dataset(dev_rows)
test_ds, test_offsets = to_dataset(test_rows)
collator = DataCollatorForTokenClassification(tokenizer)
print(train_ds)

## 7. 모델: 사전학습 본체 + 분류 헤드

`AutoModelForTokenClassification` 은 두 부분으로 되어 있다.

1. **본체 klue/roberta-base** (사전학습 완료): 12층 Transformer 인코더, 은닉 768차원, 어텐션 헤드 12개, 약 1.1억 파라미터. 한국어 문장을 대량으로 읽으며 "가린 단어 맞히기"로 문맥을 익혔다. 각 층의 셀프 어텐션은 `softmax(QKᵀ/√d)·V` 로 토큰마다 문장의 다른 토큰을 얼마나 참고할지 가중치를 계산한다. 마지막 층에서 토큰마다 **문맥이 반영된 768차원 벡터**가 나온다.
2. **분류 헤드** (새로 만듦, 무작위 초기값): 선형층 768 → 15. 토큰 벡터를 15개 꼬리표 점수로 바꾼다.

불러올 때 "일부 가중치가 새로 초기화됐다"는 경고가 나오는데, 이 분류 헤드 이야기라 정상이다.

**파인튜닝**: 헤드만이 아니라 본체 12층까지 함께 조금씩 고친다. 손실은 토큰별 교차 엔트로피(정답 꼬리표의 확률에 -log), `-100` 토큰은 제외.

In [ ]:
from transformers import AutoModelForTokenClassification, set_seed
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
n_all = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for n, p in model.named_parameters() if "classifier" in n)
print(f"전체 파라미터 {n_all:,} | 분류 헤드 {n_head:,} (= 768×15 + 15)")

## 8. 학습 설정

- **옵티마이저 AdamW**, 학습률 `5e-5`: 사전학습된 본체를 크게 망가뜨리지 않도록 작게 잡는다.
- **warmup 10%**: 처음엔 학습률을 0에서 천천히 올린다. 무작위 헤드 때문에 초반 기울기가 커서 본체가 흔들리는 것을 막는다. 이후 선형으로 줄인다.
- **weight decay 0.01**: 가중치가 너무 커지지 않게 하는 규제.
- **fp16**: GPU에서 16비트로 계산해 속도·메모리 절약.
- 매 에폭 끝에 **dev 셋**으로 seqeval 개체 F1을 잰다(학습 중 확인용). 최종 점수는 학습에 전혀 안 쓴 **test 셋**으로 따로 잰다.

In [ ]:
from transformers import TrainingArguments, Trainer
from seqeval.metrics import f1_score

def compute_metrics(p):
    logits, labels = p
    preds = logits.argmax(-1)
    true_seq, pred_seq = [], []
    for pr, lb in zip(preds, labels):
        keep = lb != -100
        true_seq.append([id2label[int(x)] for x in lb[keep]])
        pred_seq.append([id2label[int(x)] for x in pr[keep]])
    return {"seqeval_f1": f1_score(true_seq, pred_seq)}

import math
total_steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
warmup = int(0.1 * total_steps)          # 전체 스텝의 10%. 정수로 주면 transformers 4.x·5.x 모두 동작
print("전체 스텝", total_steps, "| warmup 스텝", warmup)
args = TrainingArguments(
    output_dir=f"/content/{EXP_NAME}", learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
    warmup_steps=warmup, weight_decay=0.01, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="no", logging_steps=20, report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                  data_collator=collator, compute_metrics=compute_metrics)

## 9. 학습
로그의 `loss` 가 줄고 에폭마다 `eval_seqeval_f1` 이 오르면 정상이다. 희소 태그(NW·TR·FD)는 학습 문장에 적어서 점수가 낮을 수 있다. 그것도 결과로 기록할 내용이다.

In [ ]:
import time
t0 = time.time()
trainer.train()
train_sec = round(time.time() - t0, 1)
print("학습 시간(초):", train_sec)

# 최종 가중치를 드라이브에 저장하고 다시 읽어 확인 (10/23: /content 에만 두면 런타임이 끝날 때 사라짐)
MODEL_OUT = f"/content/drive/MyDrive/ner_models/{EXP_NAME}"
trainer.save_model(MODEL_OUT); tokenizer.save_pretrained(MODEL_OUT)
from safetensors.torch import load_file
saved = load_file(f"{MODEL_OUT}/model.safetensors"); live = model.state_dict()
assert all(torch.equal(v.cpu(), live[k].detach().cpu()) for k, v in saved.items()) and len(saved) >= 0.95 * len(live), "가중치 저장 확인 실패"
print("최종 가중치 저장·확인:", MODEL_OUT, f"텐서 {len(saved)}개 일치")

## 10. 테스트 평가 (최종 점수)
테스트 문장의 예측 꼬리표를 글자 span으로 복원해 정답 span과 엄격 일치로 채점한다. 전체 점수와 태그별 점수를 함께 본다. `support` 는 그 태그의 정답 개체 수다. support가 몇 개 안 되는 태그의 점수는 우연의 영향이 커서 해석에 주의한다.

In [ ]:
t0 = time.time()
pred_out = trainer.predict(test_ds)
infer_sec = round(time.time() - t0, 1)
pred_ids = pred_out.predictions.argmax(-1)

pred_spans = []
for i, offs in enumerate(test_offsets):
    n = len(offs)
    pred_spans.append(decode_spans(pred_ids[i][:n], offs))
test_metrics = span_prf([r["spans"] for r in test_rows], pred_spans)

print(f"{'태그':4s} {'P':>7s} {'R':>7s} {'F1':>7s} {'정답수':>6s}")
for k, v in test_metrics.items():
    print(f"{k:4s} {v['precision']:7.4f} {v['recall']:7.4f} {v['f1']:7.4f} {v['support']:6d}")
print("추론 시간(초):", infer_sec)

## 11. 틀린 예시 보기
점수만 보면 왜 틀렸는지 모른다. 틀린 문장을 몇 개 보고 유형을 나눈다.
- **경계 오류**: 태그는 맞는데 범위가 다름 (예: `한국은행` vs `한국은행이`)
- **유형 오류**: 범위는 맞는데 태그가 다름
- **미탐**: 정답에 있는데 모델이 못 찾음
- **오탐**: 정답에 없는데 모델이 냄. 데이터에 빠진 개체를 모델이 맞게 찾은 경우도 여기 들어간다(라벨 노이즈).

In [ ]:
random.seed(0)
wrong = [i for i, r in enumerate(test_rows) if set(map(tuple, r["spans"])) != set(pred_spans[i])]
print(f"틀린 문장 {len(wrong)}/{len(test_rows)}")
for i in random.sample(wrong, min(5, len(wrong))):
    t = test_rows[i]["text"]; g = set(map(tuple, test_rows[i]["spans"])); p = set(pred_spans[i])
    print("\n문장:", t[:150])
    print("  정답에만:", [(t[s:e], tag) for s, e, tag in sorted(g - p)])
    print("  예측에만:", [(t[s:e], tag) for s, e, tag in sorted(p - g)])

## 12. 새 문장 넣어 보기 (시연용)
발표에 쓸 "문장 입력 → 개체 출력" 함수.

In [ ]:
def predict(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_offsets_mapping=True,
                    return_token_type_ids=False, return_tensors="pt")
    offs = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        ids = model(**enc).logits.argmax(-1)[0].tolist()
    return [(text[s:e], tag, s, e) for s, e, tag in decode_spans(ids, offs)]

for s in ["한국은행 금융통화위원회는 기준금리를 연 3.5%로 동결했다.",
          "삼성전자는 올해 반도체 부문에서 고대역폭 메모리 생산을 늘린다고 밝혔다."]:
    print(s); print("  →", predict(s))

## 13. 결과 저장
설정과 점수를 json으로 남긴다. 팀 비교표는 이 파일들을 모아서 만든다. **표에는 이렇게 실제로 잰 값만 넣는다.**

In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
result = {"exp": EXP_NAME, "model": MODEL_NAME, "data_dir": DATA_DIR,
          "n_train": len(train_rows), "n_dev": len(dev_rows), "n_test": len(test_rows),
          "max_len": MAX_LEN, "lr": LR, "epochs": EPOCHS, "batch": BATCH, "seed": SEED,
          "train_sec": train_sec, "infer_sec": infer_sec,
          "dev_log": [h for h in trainer.state.log_history if "eval_seqeval_f1" in h],
          "test_span_metrics": test_metrics}
path = f"{OUT_DIR}/{EXP_NAME}.json"
json.dump(result, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("저장:", path)

## 14. 정리와 다음 실험

**스스로 설명해 보기** (PR 설명 3줄용)
1. 왜 출력이 7개가 아니라 15개인가?
2. 토큰 꼬리표를 어떻게 글자 위치로 되돌리는가? (`offset_mapping`)
3. 왜 5-1의 상한 F1이 100%가 아닌가?

**다음 실험**: 1번 설정 셀에서 `MODEL_NAME` 과 `EXP_NAME` 만 바꿔 다시 돌린다.

| 실험 | MODEL_NAME |
|---|---|
| E2 AI Hub ELECTRA | 압축을 푼 `1세부_언어모델` 폴더 경로 (config.json, pytorch_model.bin, vocab.txt 가 있는 곳) |
| 대조군 KoELECTRA | `monologg/koelectra-base-v3-discriminator` |
| BERT 비교 | `klue/bert-base` |

학습 데이터 양 실험(1,500 / 6만 / 30만 문장)이 필요하면 `src/make_splits.py --n-train ... --out data/split_xxx` 로 새 표본을 만든 뒤 `DATA_DIR` 만 바꾼다.